In [13]:
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


In [14]:
input_path = "E:/LogAnomalyDetector/data/parsed_logs.csv"

df = pd.read_csv(input_path)

print("Loaded dataset:")
print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())


Loaded dataset:
             timestamp     level             ip
0  2025-07-23 16:11:43      INFO  192.168.1.100
1  2025-07-23 16:13:22   WARNING  192.168.1.101
2  2025-07-23 16:15:11     ERROR            NaN
3  2025-07-23 16:18:53      INFO            NaN
4  2025-07-23 16:21:08  CRITICAL  192.168.1.105

Shape: (5, 3)

Columns: ['timestamp', 'level', 'ip']


In [18]:
# Try to automatically find a suitable text column
possible_text_cols = [
    "message", "log", "raw_log", "text", "line", "event", "content", "description"
]

found_text_col = None
for col in possible_text_cols:
    if col in df.columns:
        found_text_col = col
        break

if found_text_col:
    print(f"\n[INFO] Using existing text column '{found_text_col}' as message.")
    df["message"] = df[found_text_col].astype(str)

else:
    print("\n[WARNING] No explicit text column found.")
    print("[INFO] Constructing fallback message from available fields (row-wise).")

    usable_cols = []
    for col in ["level", "ip", "service", "host", "process", "pid"]:
        if col in df.columns:
            usable_cols.append(col)

    if not usable_cols:
        raise ValueError(
            "No usable fields to construct message. "
            "Dataset is not suitable for text-based anomaly detection."
        )

    # Row-wise construction of message
def build_message(row):
    parts = []
    for col in usable_cols:
        val = row[col]
        if pd.isna(val):
            continue
        parts.append(f"{col.upper()}={val}")
    return " ".join(parts)


    df["message"] = df.apply(build_message, axis=1)

print("\nColumns after ensuring message:")
print(df.columns.tolist())

print("\nSample constructed messages:")
print(df["message"].head())



[INFO] Using existing text column 'message' as message.

Columns after ensuring message:
['timestamp', 'level', 'ip', 'message']

Sample constructed messages:
0        LEVEL=INFO IP=192.168.1.100
1     LEVEL=WARNING IP=192.168.1.101
2                 LEVEL=ERROR IP=nan
3                  LEVEL=INFO IP=nan
4    LEVEL=CRITICAL IP=192.168.1.105
Name: message, dtype: object


In [19]:
# -------------------------------
# Weak supervision labeling
# -------------------------------

def assign_label(row):
    # Rule-based proxy label
    if row.get("level") in ["ERROR", "CRITICAL"]:
        return 1
    else:
        return 0

df["true_label"] = df.apply(assign_label, axis=1)

print("\nLabel distribution:")
print(df["true_label"].value_counts())



Label distribution:
true_label
0    3
1    2
Name: count, dtype: int64


In [20]:
# -------------------------------
# Save minimal labeled dataset
# -------------------------------

final_cols = ["timestamp", "message", "true_label"]
df_final = df[final_cols]

save_path = "E:/LogAnomalyDetector/data/labeled_logs.csv"
df_final.to_csv(save_path, index=False)

print("\nSaved labeled data to:", save_path)
print("Final shape:", df_final.shape)
print("\nSample of final dataset:")
print(df_final.head())



Saved labeled data to: E:/LogAnomalyDetector/data/labeled_logs.csv
Final shape: (5, 3)

Sample of final dataset:
             timestamp                          message  true_label
0  2025-07-23 16:11:43      LEVEL=INFO IP=192.168.1.100           0
1  2025-07-23 16:13:22   LEVEL=WARNING IP=192.168.1.101           0
2  2025-07-23 16:15:11               LEVEL=ERROR IP=nan           1
3  2025-07-23 16:18:53                LEVEL=INFO IP=nan           0
4  2025-07-23 16:21:08  LEVEL=CRITICAL IP=192.168.1.105           1
